# Predictive maintenance — end-to-end pipeline

Single notebook covering all phases for every source: **ingestion →
understanding → processing → database**, then the **cross-source analysis**.
It reuses the code in `src/` (the same code the orchestrator runs) and adds
inline exploration. For a full automated run: `uv run python scripts/run_pipeline.py`.

## 0. Setup

In [ ]:
import sys, tempfile
from pathlib import Path

%load_ext autoreload
%autoreload 2

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

%matplotlib inline

from src import config
from src.common.env import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

# Scratch folder for graphs generated inline (not persisted in the repo).
SCRATCH = Path(tempfile.mkdtemp(prefix='nb_'))
print('Project root:', PROJECT_ROOT)

## 1. Ingestion (raw → DataFrame)

Each source keeps its own loader in `src/sources/<source>/loader.py`.

In [ ]:
from src.sources.incidents.loader import load_incidents
from src.sources.telemetry.loader import load_telemetry
from src.sources.machines.loader import build_engine, load_maintenance

df_incidents = load_incidents(config.DEFAULT_INPUT_CSV)
df_telemetry = load_telemetry(config.DEFAULT_TELEMETRY_CSV)
df_maintenance = load_maintenance(build_engine(config.DEFAULT_MACHINES_SQL))

for name, d in [('incidents', df_incidents), ('telemetry', df_telemetry), ('maintenance', df_maintenance)]:
    print(f'{name:12} {d.shape}')

In [ ]:
df_incidents.head()

## 2. Understanding (data reports)

For each source: **A. via `src/`** (standard graphs, generated then displayed)
and **B. inline** (ad-hoc exploration). The standard run also writes
`run_report.md` + `dataset_report.md` (see `scripts/run_pipeline.py`).

### 2.1 Incidents — A. standard graphs via `src/`

In [ ]:
from src.sources.incidents import distributions, histograms, correlations
from src.sources.incidents import runner as incidents_runner

inc = incidents_runner.load_dataframe()  # anonymised + enriched
out = SCRATCH / 'incidents'; out.mkdir(exist_ok=True)
paths = distributions.plot_all(inc, out) + histograms.plot_all(inc, out) + correlations.plot_all(inc, out)
for p in paths:
    display(Image(filename=str(p)))

### 2.1 Incidents — B. inline: incidents by severity and machine

In [ ]:
ct = pd.crosstab(inc[config.MACHINE_COLUMN], inc[config.SEVERITY_COLUMN])
ct = ct.sort_values(by=sorted(ct.columns, reverse=True), ascending=False)
display(ct)

### 2.2 Telemetry — A. standard graphs via `src/`

In [ ]:
from src.sources.telemetry import boxplots

out = SCRATCH / 'telemetry'; out.mkdir(exist_ok=True)
for p in boxplots.plot_all(df_telemetry, out):
    display(Image(filename=str(p)))

### 2.2 Telemetry — B. inline: parameter correlation

In [ ]:
params = [c for c in config.TELEMETRY_PARAM_COLUMNS if c in df_telemetry.columns]
df_telemetry[params].corr().round(2)

### 2.3 Machines — A. standard graphs via `src/`

In [ ]:
from src.sources.machines import plots as machines_plots

out = SCRATCH / 'machines'; out.mkdir(exist_ok=True)
for p in machines_plots.plot_all(df_maintenance, out):
    display(Image(filename=str(p)))

### 2.3 Machines — B. inline: duration by maintenance type

In [ ]:
df_maintenance.groupby(config.MAINTENANCE_TYPE_COLUMN)[config.MAINTENANCE_DURATION_COLUMN].agg(['count', 'mean', 'median', 'max']).round(2)

## 3. Processing (mutualised tools)

`src/processing/` applies, per source (declared in `src/sources/registry.py`):
text→value encoding, imputation and IQR outlier clipping. Anonymisation is
applied earlier (at ingestion) for PII sources.

In [ ]:
from src.sources.registry import SOURCE_SPECS
from src.processing.pipeline import apply_processing

processed = {}
for spec in SOURCE_SPECS:
    raw = spec.load_dataframe()
    proc, report = apply_processing(raw, spec.processing)
    processed[spec.name] = proc
    print(f'{spec.name:12} {raw.shape} -> {proc.shape} | steps: {list(report)}')

### Example: telemetry outlier treatment (before / after)

In [ ]:
spec = next(s for s in SOURCE_SPECS if s.name == 'telemetry')
raw = spec.load_dataframe()
proc, report = apply_processing(raw, spec.processing)
display(report['outliers'])
pd.DataFrame({'before': raw['temperature_c'].describe(), 'after': proc['temperature_c'].describe()}).round(2)

## 4. Loading to PostgreSQL

The orchestrator (`scripts/run_pipeline.py`) loads each processed dataset into a
table (full reload). Here we just **read back** the tables to confirm. Start the
DB with `docker compose up -d` if needed.

In [ ]:
from sqlalchemy import text
from src.database.engine import is_available, get_engine

if is_available():
    engine = get_engine()
    with engine.connect() as conn:
        for table in ['incidents', 'telemetry', 'maintenance']:
            n = conn.execute(text(f'SELECT count(*) FROM "{table}"')).scalar()
            print(f'{table:12} {n} rows')
else:
    print('PostgreSQL unavailable — run: docker compose up -d, then scripts/run_pipeline.py')

## 5. Cross-source analysis

**A. via `src/`** (joined graphs) and **B. inline** (correlation matrix of the
per-machine profile).

In [ ]:
from src.analyses.joins import load_sources, build_machine_profile, build_reactive_incident_join
from src.analyses import plots as cross_plots

sources = load_sources()
profile = build_machine_profile(sources)
reactive = build_reactive_incident_join(sources)

out = SCRATCH / 'cross'; out.mkdir(exist_ok=True)
for p in cross_plots.plot_all(profile, reactive, out):
    display(Image(filename=str(p)))

### 5. Cross-source — B. inline: machine profile correlation matrix

In [ ]:
num = profile.select_dtypes('number')
corr = num.corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index))); ax.set_yticklabels(corr.index)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Machine profile — correlation matrix')
plt.tight_layout(); plt.show()

---
Reminder: this notebook is for exploration. The reproducible pipeline is
`scripts/run_pipeline.py` (re-run it after any data update in `data/raw/`).